In [1]:
#cell 1
# Install runtime dependencies for Colab + Blackwell + vLLM.

!pip -q install -U uv

!uv pip install --system -U openai requests tqdm jsonschema psutil numpy pandas accelerate safetensors huggingface_hub

# Remove optional packages that may break transformers/vLLM imports in Colab.
!uv pip uninstall --system -y torchcodec torchvision torchaudio sentence-transformers || true

# Transformers is used for:
# 1) LLM tokenizer-based input truncation
# 2) lightweight CPU query encoder for all-MiniLM-L6-v2
!uv pip install --system -U "transformers>=4.51.0"

# Recent vLLM nightly for CUDA 13 / Blackwell.
!uv pip install --system -U vllm --torch-backend=cu130 --extra-index-url https://wheels.vllm.ai/nightly/cu130

# Fallback only if the cu130 line fails:
# !uv pip install --system -U vllm --torch-backend=auto --extra-index-url https://wheels.vllm.ai/nightly

import sys
import importlib.metadata as md

import torch
import vllm
import transformers

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("vLLM:", vllm.__version__)
print("transformers:", transformers.__version__)

for pkg in ["torchcodec", "torchvision", "torchaudio", "sentence-transformers"]:
    try:
        print(pkg + ":", md.version(pkg))
    except Exception:
        print(pkg + ": not installed")

!nvidia-smi

Using Python 3.12.13 environment at: /usr
Resolved 71 packages in 224ms
Prepared 8 packages in 0.34ms
Uninstalled 8 packages in 122ms
Installed 8 packages in 108ms
 - numpy==2.3.5
 + numpy==2.5.0
 - nvidia-cublas==13.1.0.3
 + nvidia-cublas==13.1.1.3
 - nvidia-cudnn-cu13==9.19.0.56
 + nvidia-cudnn-cu13==9.20.0.48
 - nvidia-cusparselt-cu13==0.8.0
 + nvidia-cusparselt-cu13==0.8.1
 - nvidia-nccl-cu13==2.28.9
 + nvidia-nccl-cu13==2.29.7
 - setuptools==80.10.2
 + setuptools==81.0.0
 - torch==2.11.0+cu130
 + torch==2.12.1
 - triton==3.6.0
 + triton==3.7.1
Using Python 3.12.13 environment at: /usr
Uninstalled 2 packages in 48ms
 - torchaudio==2.11.0+cu130
 - torchvision==0.26.0+cu130
Using Python 3.12.13 environment at: /usr
Resolved 27 packages in 178ms
Checked 27 packages in 0.29ms
Using Python 3.12.13 environment at: /usr
Resolved 191 packages in 3.90s
Prepared 10 packages in 27ms
Uninstalled 8 packages in 111ms
Installed 10 packages in 117ms
 - numpy==2.5.0
 + numpy==2.3.5
 - nvidia-cublas

In [2]:
#cell 2
# Imports and global configuration.

import os
import re
import gc
import ast
import json
import time
import shlex
import shutil
import random
import psutil
import subprocess
import traceback
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
from openai import OpenAI
from transformers import AutoTokenizer, AutoModel

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Keep CPU query encoding separate from the vLLM GPU server.
torch.set_num_threads(max(1, (psutil.cpu_count(logical=True) or 2) - 1))

LLM_MODEL_NAME = "Qwen/Qwen3.5-27B"
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

PORT = 8000
BASE_URL = f"http://localhost:{PORT}/v1"

# LogicRAG baseline settings requested by the user.
TOP_K = 5
MAX_ROUNDS = 3

# Token budgets with safety margin for ~512-token chunks and rolling memory.
MAX_MODEL_LEN = 16384
MAX_INPUT_TOKENS = 14336
SUMMARY_MAX_TOKENS = 768
JSON_MAX_TOKENS = 512
ANSWER_MAX_TOKENS = 384

GPU_MEMORY_UTILIZATION = 0.92
MAX_NUM_SEQS = 1
MAX_NUM_BATCHED_TOKENS = MAX_MODEL_LEN

SERVER_LOG_PATH = Path("/content/vllm_logicrag_qwen35_server.log")
SERVER_PID_PATH = Path("/content/vllm_logicrag_qwen35_server.pid")

DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE_ROOT / "final_project"
WORK_DIR = PROJECT_DIR / "logicRAG"

DATA_DIR = WORK_DIR / "data"
EMBEDDINGS_DIR = WORK_DIR / "embeddings"
CACHE_DIR = WORK_DIR / "cache"

DRIVE_EVIDENCE_DIR = WORK_DIR / "evidence"
DRIVE_ANSWER_DIR = WORK_DIR / "answer"

LOCAL_RUNTIME_DIR = Path("/content/final_project_logicrag_stage2")
LOCAL_QUESTION_DIR = LOCAL_RUNTIME_DIR / "questions"
LOCAL_CORPUS_DIR = LOCAL_RUNTIME_DIR / "corpus"
LOCAL_EMBEDDING_DIR = LOCAL_RUNTIME_DIR / "embeddings"

for path in [
    LOCAL_RUNTIME_DIR,
    LOCAL_QUESTION_DIR,
    LOCAL_CORPUS_DIR,
    LOCAL_EMBEDDING_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

SAVE_EVERY_N = 10
CLEAR_CACHE_EVERY_N = 25
SHUFFLE_SEED = 3407

# Use None to process all shuffled records.
QUESTION_START_INDEX = 0
QUESTION_END_INDEX = None

RESUME_IF_EXISTS = True
EXPECTED_NUM_RECORDS_PER_DATASET = 1000

DATASETS = {
    "hotpotqa": {
        "question_drive_path": PROJECT_DIR / "hotpotqa_dev_2017wiki_1000_converted.json",
        "corpus_drive_path": DATA_DIR / "hotpotqa_logicrag_corpus.json",
        "manifest_drive_path": EMBEDDINGS_DIR / "hotpot_manifest.json",
        "question_local_original_path": LOCAL_QUESTION_DIR / "hotpotqa_questions_original.json",
        "question_local_shuffled_path": LOCAL_QUESTION_DIR / "hotpotqa_questions_shuffled.json",
        "corpus_local_path": LOCAL_CORPUS_DIR / "hotpotqa_logicrag_corpus.json",
        "embedding_local_path": LOCAL_EMBEDDING_DIR / "hotpotqa_embeddings.pt",
        "evidence_output_path": DRIVE_EVIDENCE_DIR / "hotpotqa_evidence.json",
        "answer_output_path": DRIVE_ANSWER_DIR / "hotpotqa_qwen3.5_answers.json",
    },
    "2wikimultihopqa": {
        "question_drive_path": PROJECT_DIR / "2wikimultihopqa_dev_2020wiki_1000_converted.json",
        "corpus_drive_path": DATA_DIR / "2wikimultihopqa_logicrag_corpus.json",
        "manifest_drive_path": EMBEDDINGS_DIR / "2wiki_manifest.json",
        "question_local_original_path": LOCAL_QUESTION_DIR / "2wikimultihopqa_questions_original.json",
        "question_local_shuffled_path": LOCAL_QUESTION_DIR / "2wikimultihopqa_questions_shuffled.json",
        "corpus_local_path": LOCAL_CORPUS_DIR / "2wikimultihopqa_logicrag_corpus.json",
        "embedding_local_path": LOCAL_EMBEDDING_DIR / "2wikimultihopqa_embeddings.pt",
        "evidence_output_path": DRIVE_EVIDENCE_DIR / "2wikimultihopqa_evidence.json",
        "answer_output_path": DRIVE_ANSWER_DIR / "2wikimultihopqa_qwen3.5_answers.json",
    },
}

DATASET_RUN_ORDER = ["hotpotqa", "2wikimultihopqa"]

print("LLM model:", LLM_MODEL_NAME)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Top-k:", TOP_K)
print("Max rounds:", MAX_ROUNDS)
print("Max model len:", MAX_MODEL_LEN)
print("Max input tokens:", MAX_INPUT_TOKENS)
print("Save every N questions:", SAVE_EVERY_N)
print("Work dir:", WORK_DIR)
print("Evidence output dir:", DRIVE_EVIDENCE_DIR)
print("Answer output dir:", DRIVE_ANSWER_DIR)

for dataset_name, cfg in DATASETS.items():
    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Question file:", cfg["question_drive_path"])
    print("Corpus file:", cfg["corpus_drive_path"])
    print("Manifest file:", cfg["manifest_drive_path"])
    print("Evidence output:", cfg["evidence_output_path"])
    print("Answer output:", cfg["answer_output_path"])

LLM model: Qwen/Qwen3.5-27B
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Top-k: 5
Max rounds: 3
Max model len: 16384
Max input tokens: 14336
Save every N questions: 10
Work dir: /content/drive/MyDrive/final_project/logicRAG
Evidence output dir: /content/drive/MyDrive/final_project/logicRAG/evidence
Answer output dir: /content/drive/MyDrive/final_project/logicRAG/answer
Dataset: hotpotqa
Question file: /content/drive/MyDrive/final_project/hotpotqa_dev_2017wiki_1000_converted.json
Corpus file: /content/drive/MyDrive/final_project/logicRAG/data/hotpotqa_logicrag_corpus.json
Manifest file: /content/drive/MyDrive/final_project/logicRAG/embeddings/hotpot_manifest.json
Evidence output: /content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json
Answer output: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_qwen3.5_answers.json
Dataset: 2wikimultihopqa
Question file: /content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json

In [3]:
#cell 3
# Mount Google Drive and copy corpus/embedding files to local Colab disk.

from google.colab import drive

MOUNTPOINT = Path("/content/drive")
drive.mount(str(MOUNTPOINT), force_remount=True)

assert PROJECT_DIR.exists(), f"PROJECT_DIR does not exist: {PROJECT_DIR}"
assert WORK_DIR.exists(), f"WORK_DIR does not exist: {WORK_DIR}"
assert DATA_DIR.exists(), f"DATA_DIR does not exist: {DATA_DIR}"
assert EMBEDDINGS_DIR.exists(), f"EMBEDDINGS_DIR does not exist: {EMBEDDINGS_DIR}"

DRIVE_EVIDENCE_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ANSWER_DIR.mkdir(parents=True, exist_ok=True)

def file_is_same_size(src: Path, dst: Path) -> bool:
    """Check whether the destination file exists and has the same size as the source."""
    return dst.exists() and dst.stat().st_size == src.stat().st_size

def copy_file_to_local(src: Path, dst: Path) -> None:
    """Copy a file to local disk using a temporary file to avoid partial copies."""
    dst.parent.mkdir(parents=True, exist_ok=True)

    if file_is_same_size(src, dst):
        print(f"Local copy already exists: {dst}")
        return

    tmp = dst.with_name(dst.name + ".tmp")
    if tmp.exists():
        tmp.unlink()

    shutil.copy2(src, tmp)
    os.replace(tmp, dst)
    print(f"Copied to local disk: {src} -> {dst}")

def load_json(path: Path) -> Any:
    """Load UTF-8 JSON."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

for dataset_name, cfg in DATASETS.items():
    print("=" * 100)
    print("Preparing dataset:", dataset_name)

    for required_path in [
        cfg["question_drive_path"],
        cfg["corpus_drive_path"],
        cfg["manifest_drive_path"],
    ]:
        if not required_path.exists():
            raise FileNotFoundError(f"Missing required file: {required_path}")

    manifest = load_json(cfg["manifest_drive_path"])
    embedding_drive_path = Path(manifest["repository_cache_file"])

    if not embedding_drive_path.exists():
        raise FileNotFoundError(
            f"{dataset_name}: embedding cache from manifest does not exist: {embedding_drive_path}"
        )

    cfg["embedding_drive_path"] = embedding_drive_path

    copy_file_to_local(cfg["corpus_drive_path"], cfg["corpus_local_path"])
    copy_file_to_local(embedding_drive_path, cfg["embedding_local_path"])

    print("Corpus local size MB:", cfg["corpus_local_path"].stat().st_size / (1024 ** 2))
    print("Embedding local size MB:", cfg["embedding_local_path"].stat().st_size / (1024 ** 2))
    print("Embedding source:", embedding_drive_path)

print("Drive output directories are ready.")

Mounted at /content/drive
Preparing dataset: hotpotqa
Copied to local disk: /content/drive/MyDrive/final_project/logicRAG/data/hotpotqa_logicrag_corpus.json -> /content/final_project_logicrag_stage2/corpus/hotpotqa_logicrag_corpus.json
Copied to local disk: /content/drive/MyDrive/final_project/logicRAG/cache/hotpot/embeddings_35029.pt -> /content/final_project_logicrag_stage2/embeddings/hotpotqa_embeddings.pt
Corpus local size MB: 64.61611366271973
Embedding local size MB: 51.313575744628906
Embedding source: /content/drive/MyDrive/final_project/logicRAG/cache/hotpot/embeddings_35029.pt
Preparing dataset: 2wikimultihopqa
Copied to local disk: /content/drive/MyDrive/final_project/logicRAG/data/2wikimultihopqa_logicrag_corpus.json -> /content/final_project_logicrag_stage2/corpus/2wikimultihopqa_logicrag_corpus.json
Copied to local disk: /content/drive/MyDrive/final_project/logicRAG/cache/2wiki/embeddings_12685.pt -> /content/final_project_logicrag_stage2/embeddings/2wikimultihopqa_embedd

In [4]:
#cell 4
# Copy question files to local Colab disk and create deterministic shuffled copies.
# IMPORTANT: During generation, only item["question"] is passed to the LLM.
# Other keys are used only after generation for saving metadata and ground truth.

def validate_question_record(record: Dict[str, Any], dataset_name: str, index: int) -> None:
    """Validate one question record."""
    if not isinstance(record, dict):
        raise ValueError(f"{dataset_name}: record {index} is not a dictionary.")

    if "question" not in record or not isinstance(record["question"], str) or not record["question"].strip():
        raise ValueError(f"{dataset_name}: record {index} has an invalid question.")

    if "answer" not in record:
        raise ValueError(f"{dataset_name}: record {index} is missing answer.")

    if "type" not in record:
        raise ValueError(f"{dataset_name}: record {index} is missing type.")

    if "supports" not in record:
        raise ValueError(f"{dataset_name}: record {index} is missing supports.")

def load_validate_shuffle_questions(dataset_name: str, cfg: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Copy, load, validate, shuffle, and save the question records."""
    copy_file_to_local(cfg["question_drive_path"], cfg["question_local_original_path"])

    rows = load_json(cfg["question_local_original_path"])
    if not isinstance(rows, list):
        raise ValueError(f"{dataset_name}: question JSON root must be a list.")

    if len(rows) != EXPECTED_NUM_RECORDS_PER_DATASET:
        print(
            f"Warning: {dataset_name} has {len(rows)} records, "
            f"expected {EXPECTED_NUM_RECORDS_PER_DATASET}."
        )

    for i, row in enumerate(rows):
        validate_question_record(row, dataset_name, i)

    rng = random.Random(SHUFFLE_SEED)
    shuffled = list(rows)
    rng.shuffle(shuffled)

    with open(cfg["question_local_shuffled_path"], "w", encoding="utf-8") as f:
        json.dump(shuffled, f, ensure_ascii=False, indent=2)

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Original local questions:", cfg["question_local_original_path"])
    print("Shuffled local questions:", cfg["question_local_shuffled_path"])
    print("Shuffle seed:", SHUFFLE_SEED)
    print("Number of shuffled records:", len(shuffled))
    print("First shuffled question:", shuffled[0]["question"])
    print("First shuffled type:", shuffled[0]["type"])
    print("First shuffled GT answer:", shuffled[0]["answer"])

    return shuffled

shuffled_questions = {}

for dataset_name, cfg in DATASETS.items():
    shuffled_questions[dataset_name] = load_validate_shuffle_questions(dataset_name, cfg)

Copied to local disk: /content/drive/MyDrive/final_project/hotpotqa_dev_2017wiki_1000_converted.json -> /content/final_project_logicrag_stage2/questions/hotpotqa_questions_original.json
Dataset: hotpotqa
Original local questions: /content/final_project_logicrag_stage2/questions/hotpotqa_questions_original.json
Shuffled local questions: /content/final_project_logicrag_stage2/questions/hotpotqa_questions_shuffled.json
Shuffle seed: 3407
Number of shuffled records: 1000
First shuffled question: George Gershwin is an American Composer and Judith Weir is a composer from which country?
First shuffled type: comparison
First shuffled GT answer: a British composer
Copied to local disk: /content/drive/MyDrive/final_project/2wikimultihopqa_dev_2020wiki_1000_converted.json -> /content/final_project_logicrag_stage2/questions/2wikimultihopqa_questions_original.json
Dataset: 2wikimultihopqa
Original local questions: /content/final_project_logicrag_stage2/questions/2wikimultihopqa_questions_original.j

In [5]:
#cell 5
# Start Qwen3.5 vLLM server in non-thinking mode.

def kill_process_tree(pid: int) -> None:
    """Kill a process and all child processes."""
    try:
        parent = psutil.Process(int(pid))
        for child in parent.children(recursive=True):
            try:
                child.kill()
            except Exception:
                pass
        parent.kill()
        parent.wait(timeout=10)
        print("Killed process tree:", pid)
    except Exception:
        pass

# Stop old PID from this notebook.
if SERVER_PID_PATH.exists():
    old_pid = SERVER_PID_PATH.read_text().strip()
    if old_pid:
        kill_process_tree(int(old_pid))

# Stop leftover vLLM serve processes.
for p in psutil.process_iter(["pid", "name", "cmdline"]):
    try:
        cmdline = " ".join(p.info.get("cmdline") or [])
        if "vllm" in cmdline and "serve" in cmdline:
            kill_process_tree(p.info["pid"])
            print("Killed leftover vLLM process:", p.info["pid"])
    except Exception:
        pass

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

time.sleep(3)

cmd = [
    "vllm", "serve", LLM_MODEL_NAME,

    "--host", "0.0.0.0",
    "--port", str(PORT),

    "--max-model-len", str(MAX_MODEL_LEN),
    "--gpu-memory-utilization", str(GPU_MEMORY_UTILIZATION),

    # Text-only mode to skip multimodal parts and free memory.
    "--language-model-only",

    # Non-thinking mode for Qwen3/Qwen3.5.
    "--reasoning-parser", "qwen3",
    "--default-chat-template-kwargs", '{"enable_thinking": false}',

    "--max-num-seqs", str(MAX_NUM_SEQS),
    "--max-num-batched-tokens", str(MAX_NUM_BATCHED_TOKENS),

    "--enable-prefix-caching",
    "--generation-config", "vllm",
    "--dtype", "bfloat16",
    "--trust-remote-code",
]

server_env = os.environ.copy()

# Blackwell / CUDA 13 fixes.
server_env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
server_env["VLLM_MAIN_CUDA_VERSION"] = "13.0"
server_env["TORCH_CUDA_ARCH_LIST"] = "12.0"

print("Command:")
print(" ".join(shlex.quote(x) for x in cmd))

print("\nImportant environment variables:")
for k in ["VLLM_USE_FLASHINFER_SAMPLER", "VLLM_MAIN_CUDA_VERSION", "TORCH_CUDA_ARCH_LIST"]:
    print(f"{k}={server_env.get(k)}")

SERVER_LOG_PATH.write_text("", encoding="utf-8")
log_file = open(SERVER_LOG_PATH, "w", encoding="utf-8")

proc = subprocess.Popen(
    cmd,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    text=True,
    env=server_env,
)

SERVER_PID_PATH.write_text(str(proc.pid))

print("\nStarted vLLM server.")
print("PID:", proc.pid)
print("Log:", SERVER_LOG_PATH)

Command:
vllm serve Qwen/Qwen3.5-27B --host 0.0.0.0 --port 8000 --max-model-len 16384 --gpu-memory-utilization 0.92 --language-model-only --reasoning-parser qwen3 --default-chat-template-kwargs '{"enable_thinking": false}' --max-num-seqs 1 --max-num-batched-tokens 16384 --enable-prefix-caching --generation-config vllm --dtype bfloat16 --trust-remote-code

Important environment variables:
VLLM_USE_FLASHINFER_SAMPLER=0
VLLM_MAIN_CUDA_VERSION=13.0
TORCH_CUDA_ARCH_LIST=12.0

Started vLLM server.
PID: 2371
Log: /content/vllm_logicrag_qwen35_server.log


In [6]:
#cell 6
# Wait for vLLM server and create an OpenAI-compatible client.

def tail_log(path: Path, n: int = 80) -> str:
    """Read the last n log lines."""
    if not path.exists():
        return ""
    lines = path.read_text(errors="ignore").splitlines()
    return "\n".join(lines[-n:])

ready = False
SERVER_MODEL_ID = None

MAX_WAIT_SEC = 1800
SLEEP_SEC = 5
PRINT_EVERY_SEC = 60

start = time.perf_counter()
last_print = -PRINT_EVERY_SEC

for step in range(MAX_WAIT_SEC // SLEEP_SEC):
    elapsed = int(time.perf_counter() - start)

    try:
        server_pid = int(SERVER_PID_PATH.read_text().strip())
        if not psutil.pid_exists(server_pid):
            print("\n=== Last vLLM log lines ===")
            print(tail_log(SERVER_LOG_PATH, n=160))
            raise RuntimeError("vLLM server PID no longer exists.")
    except Exception as e:
        print("\n=== Last vLLM log lines ===")
        print(tail_log(SERVER_LOG_PATH, n=160))
        raise RuntimeError(f"Could not validate vLLM server PID: {e}")

    try:
        h = requests.get(f"http://localhost:{PORT}/health", timeout=5)
        if h.status_code == 200:
            m = requests.get(f"{BASE_URL}/models", timeout=10)
            if m.status_code == 200:
                ready = True
                model_info = m.json()["data"][0]
                SERVER_MODEL_ID = model_info["id"]
                print("vLLM server is ready.")
                print("Model:", SERVER_MODEL_ID)
                print("Max model len:", model_info.get("max_model_len"))
                break
    except Exception:
        pass

    if elapsed - last_print >= PRINT_EVERY_SEC:
        last_print = elapsed
        print(f"Waiting... {elapsed}s")
        recent = tail_log(SERVER_LOG_PATH, n=12)
        if recent.strip():
            print(recent)
        print("-" * 100)

    time.sleep(SLEEP_SEC)

if not ready:
    print("\n=== Last vLLM log lines ===")
    print(tail_log(SERVER_LOG_PATH, n=160))
    raise RuntimeError("vLLM server did not become ready before timeout.")

client = OpenAI(
    api_key="EMPTY",
    base_url=BASE_URL,
    timeout=3600,
)

# Deterministic decoding is better for QA baseline comparison.
LLM_SAMPLING_KWARGS = {
    "temperature": 0.0,
    "top_p": 1.0,
    "presence_penalty": 0.0,
}

LLM_EXTRA_BODY = {
    "top_k": 20,
    "min_p": 0.0,
    "repetition_penalty": 1.0,
    "chat_template_kwargs": {
        "enable_thinking": False,
    },
}

llm_tokenizer = AutoTokenizer.from_pretrained(
    LLM_MODEL_NAME,
    trust_remote_code=True,
)

print("OpenAI-compatible client is ready.")
print("Server model id:", SERVER_MODEL_ID)
print("LLM tokenizer loaded.")

Waiting... 0s
----------------------------------------------------------------------------------------------------
Waiting... 60s
(EngineCore pid=2848) INFO 07-02 05:33:00 [registry.py:134] All limits of multimodal modalities supported by the model are set to 0, running in text-only mode.
(EngineCore pid=2848) INFO 07-02 05:33:00 [parallel_state.py:1588] world_size=1 rank=0 local_rank=0 distributed_init_method=tcp://172.28.0.12:39267 backend=nccl
(EngineCore pid=2848) INFO 07-02 05:33:00 [parallel_state.py:1923] rank 0 in world size 1 is assigned as DP rank 0, PP rank 0, PCP rank 0, TP rank 0, EP rank N/A, EPLB rank N/A
(EngineCore pid=2848) INFO 07-02 05:33:00 [topk_topp_sampler.py:39] FlashInfer top-p/top-k sampling disabled via VLLM_USE_FLASHINFER_SAMPLER=0.
(EngineCore pid=2848) INFO 07-02 05:33:00 [gpu_model_runner.py:5159] Starting to load model Qwen/Qwen3.5-27B...
(EngineCore pid=2848) INFO 07-02 05:33:01 [cuda.py:542] Using backend AttentionBackendEnum.FLASH_ATTN for vit attent

In [7]:
#cell 7
# General helper functions: atomic JSON saving, token truncation, LLM calls, and JSON parsing.

def atomic_save_json(obj: Any, path: Path) -> None:
    """Atomically save JSON to avoid corrupt partial files."""
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    os.replace(tmp, path)

def load_json_if_exists(path: Path, default: Any) -> Any:
    """Load JSON if it exists; otherwise return default."""
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def strip_thinking_blocks(text: str) -> str:
    """Remove possible Qwen thinking blocks if any appear despite non-thinking mode."""
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL | re.IGNORECASE)
    text = text.replace("<think>", "").replace("</think>", "")
    return text.strip()

def clean_final_answer(text: str) -> str:
    """Clean the final answer text without changing its meaning."""
    text = strip_thinking_blocks(text)
    text = re.sub(r"^\s*(ans|answer)\s*:\s*", "", text, flags=re.IGNORECASE).strip()
    return text

def count_llm_tokens(text: str) -> int:
    """Count tokens using the LLM tokenizer."""
    return len(llm_tokenizer.encode(text, add_special_tokens=False))

def truncate_to_tokens(text: str, max_tokens: int) -> str:
    """Truncate text to a token budget while preserving the beginning and the end."""
    if not isinstance(text, str):
        text = str(text)

    ids = llm_tokenizer.encode(text, add_special_tokens=False)
    if len(ids) <= max_tokens:
        return text

    if max_tokens <= 128:
        return llm_tokenizer.decode(ids[:max_tokens], skip_special_tokens=True)

    head_tokens = min(1024, max_tokens // 4)
    tail_tokens = max_tokens - head_tokens
    kept = ids[:head_tokens] + ids[-tail_tokens:]

    return llm_tokenizer.decode(kept, skip_special_tokens=True)

def llm_chat(prompt: str, max_tokens: int, expect_json: bool = False) -> str:
    """Call the local vLLM OpenAI-compatible chat endpoint."""
    prompt = truncate_to_tokens(prompt, MAX_INPUT_TOKENS)

    system_message = "You are a helpful assistant. Do not think step by step. Do not reveal hidden reasoning."
    if expect_json:
        system_message += " Return only one valid JSON object and no markdown."

    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt},
    ]

    last_error = None
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=SERVER_MODEL_ID,
                messages=messages,
                max_tokens=max_tokens,
                **LLM_SAMPLING_KWARGS,
                extra_body=LLM_EXTRA_BODY,
            )
            content = response.choices[0].message.content or ""
            return strip_thinking_blocks(content)
        except Exception as e:
            last_error = e
            print(f"LLM call failed on attempt {attempt + 1}/3: {e}")
            time.sleep(5 * (attempt + 1))

    raise RuntimeError(f"LLM call failed after retries: {last_error}")

def parse_json_object(text: str) -> Optional[Dict[str, Any]]:
    """Parse a JSON-like object from model output."""
    if not isinstance(text, str):
        return None

    cleaned = strip_thinking_blocks(text)
    cleaned = cleaned.replace("```json", "").replace("```", "").strip()

    # First try strict JSON.
    try:
        obj = json.loads(cleaned)
        return obj if isinstance(obj, dict) else None
    except Exception:
        pass

    # Extract the largest object-looking substring.
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start >= 0 and end > start:
        obj_text = cleaned[start:end + 1]

        try:
            obj = json.loads(obj_text)
            return obj if isinstance(obj, dict) else None
        except Exception:
            pass

        # Some models return Python-like tuples for dependency_pairs.
        try:
            obj = ast.literal_eval(obj_text)
            return obj if isinstance(obj, dict) else None
        except Exception:
            pass

        # Fix a missing final brace.
        try:
            obj = json.loads(obj_text + "}")
            return obj if isinstance(obj, dict) else None
        except Exception:
            pass

    return None

def coerce_bool(value: Any, default: bool = False) -> bool:
    """Convert bool-like model outputs to Python bool."""
    if isinstance(value, bool):
        return value

    if isinstance(value, str):
        v = value.strip().lower()
        if v in {"true", "yes", "y", "1"}:
            return True
        if v in {"false", "no", "n", "0"}:
            return False

    if isinstance(value, (int, float)):
        return bool(value)

    return default

def normalize_dependencies(value: Any, fallback_question: str) -> List[str]:
    """Normalize the dependency list returned by the LLM."""
    if isinstance(value, list):
        deps = [str(x).strip() for x in value if str(x).strip()]
    elif isinstance(value, str) and value.strip():
        deps = [value.strip()]
    else:
        deps = []

    return deps if deps else [fallback_question]

def normalize_dependency_pairs(value: Any) -> List[Tuple[int, int]]:
    """Normalize dependency pairs returned as lists, tuples, or dictionaries."""
    pairs = []

    if not isinstance(value, list):
        return pairs

    for item in value:
        try:
            if isinstance(item, dict):
                a = item.get("dependent_idx", item.get("dependent", item.get("a")))
                b = item.get("dependency_idx", item.get("dependency", item.get("b")))
            elif isinstance(item, (list, tuple)) and len(item) >= 2:
                a, b = item[0], item[1]
            else:
                continue

            pairs.append((int(a), int(b)))
        except Exception:
            continue

    return pairs

print("Helper functions are ready.")

Helper functions are ready.


In [8]:
#cell 8
# Lightweight CPU query encoder and retrieval index.
# Corpus embeddings are loaded from the stage-1 LogicRAG embedding outputs.
# The repository uses cosine similarity, so this class normalizes in memory only for fast cosine ranking.

class MiniLMQueryEncoder:
    """CPU query encoder equivalent to the pooling used by all-MiniLM-L6-v2 SentenceTransformer."""

    def __init__(self, model_name: str = EMBEDDING_MODEL_NAME, device: str = "cpu"):
        self.model_name = model_name
        self.device = torch.device(device)

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name)
        self.model.to(self.device)
        self.model.eval()

        # sentence-transformers/all-MiniLM-L6-v2 uses 256 as max sequence length.
        self.max_length = 256

        print("Query encoder loaded:", model_name)
        print("Query encoder device:", self.device)
        print("Query encoder max length:", self.max_length)

    @torch.inference_mode()
    def encode(self, texts: List[str], batch_size: int = 16) -> torch.Tensor:
        """Encode one or more query strings into sentence embeddings."""
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch = texts[i:i + batch_size]

            encoded = self.tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=self.max_length,
                return_tensors="pt",
            )
            encoded = {k: v.to(self.device) for k, v in encoded.items()}

            outputs = self.model(**encoded)
            token_embeddings = outputs.last_hidden_state
            attention_mask = encoded["attention_mask"]

            input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
            sentence_embeddings = torch.sum(token_embeddings * input_mask_expanded, dim=1)
            sentence_embeddings = sentence_embeddings / torch.clamp(
                input_mask_expanded.sum(dim=1),
                min=1e-9,
            )

            all_embeddings.append(sentence_embeddings.cpu())

        return torch.cat(all_embeddings, dim=0)

class Stage1EmbeddingRetriever:
    """Repository-compatible retrieval using the saved stage-1 embeddings."""

    def __init__(
        self,
        corpus_path: Path,
        embedding_path: Path,
        query_encoder: MiniLMQueryEncoder,
        top_k: int = TOP_K,
    ):
        self.corpus_path = Path(corpus_path)
        self.embedding_path = Path(embedding_path)
        self.query_encoder = query_encoder
        self.top_k = int(top_k)
        self.retrieval_cache = {}

        with open(self.corpus_path, "r", encoding="utf-8") as f:
            self.corpus_rows = json.load(f)

        if not isinstance(self.corpus_rows, list):
            raise ValueError(f"Corpus root must be a list: {self.corpus_path}")

        self.corpus_texts = [
            f"Title: {row['title']}. Content: {row['text']}"
            for row in self.corpus_rows
        ]

        embeddings = torch.load(self.embedding_path, map_location="cpu")

        if not isinstance(embeddings, torch.Tensor):
            raise TypeError(f"Embedding file must contain a torch.Tensor: {self.embedding_path}")

        if embeddings.shape[0] != len(self.corpus_rows):
            raise ValueError(
                f"Embedding/corpus length mismatch: embeddings={embeddings.shape[0]}, "
                f"corpus={len(self.corpus_rows)}"
            )

        self.embedding_index = F.normalize(embeddings.float(), p=2, dim=1).contiguous()

        print("=" * 100)
        print("Retriever initialized.")
        print("Corpus:", self.corpus_path)
        print("Embedding:", self.embedding_path)
        print("Number of chunks:", len(self.corpus_rows))
        print("Embedding shape:", tuple(embeddings.shape))
        print("Top-k:", self.top_k)

        del embeddings
        gc.collect()

    def retrieve(self, query: str, top_k: Optional[int] = None) -> List[Dict[str, Any]]:
        """Retrieve top-k chunks for a query."""
        k = int(top_k or self.top_k)
        cache_key = (query, k)

        if cache_key in self.retrieval_cache:
            return [dict(x) for x in self.retrieval_cache[cache_key]]

        query_embedding = self.query_encoder.encode([query])[0].float()
        query_embedding = F.normalize(query_embedding.unsqueeze(0), p=2, dim=1).squeeze(0)

        scores = torch.mv(self.embedding_index, query_embedding)
        top_scores, top_indices = torch.topk(scores, k=min(k, len(self.corpus_rows)))

        results = []
        for rank, idx in enumerate(top_indices.tolist(), start=1):
            row = self.corpus_rows[idx]
            results.append({
                "title": row.get("title", ""),
                "text": row.get("text", ""),
                "chunk_id": row.get("chunk_id", str(idx)),
                "source_index": row.get("source_index", idx),
                "paragraph_id": row.get("paragraph_id", None),
                "token_count": row.get("token_count", None),
                "score": float(top_scores[rank - 1]),
                "rank": rank,
            })

        self.retrieval_cache[cache_key] = [dict(x) for x in results]
        return results

def format_chunks_for_llm(chunks: List[Dict[str, Any]]) -> str:
    """Format retrieved chunks exactly like the repository corpus strings."""
    return "\n".join(
        f"Title: {chunk.get('title', '')}. Content: {chunk.get('text', '')}"
        for chunk in chunks
    )

query_encoder = MiniLMQueryEncoder(device="cpu")

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query encoder loaded: sentence-transformers/all-MiniLM-L6-v2
Query encoder device: cpu
Query encoder max length: 256


In [9]:
#cell 9
# LogicRAG implemented with local vLLM and precomputed stage-1 embeddings.
# This follows the repository pipeline:
# warm-up retrieval -> summary memory -> warm-up analysis ->
# dependency sorting -> iterative dependency-aware retrieval -> final answer.
#
# IMPORTANT:
# - evidence_chunk stores raw retrieved chunks for later evidence exact-match analysis.
# - final_evidence_memory stores the final summarized/refined memory that is passed
#   to the LLM for final answer generation.

class LogicRAGVLLM:
    """LogicRAG baseline using local vLLM instead of OpenAI API."""

    def __init__(
        self,
        retriever: Stage1EmbeddingRetriever,
        top_k: int = TOP_K,
        max_rounds: int = MAX_ROUNDS,
    ):
        self.retriever = retriever
        self.top_k = int(top_k)
        self.max_rounds = int(max_rounds)
        self.MODEL_NAME = "LogicRAG-vLLM-Qwen3.5"
        self.last_dependency_analysis = []

    def refine_summary_with_context(
        self,
        question: str,
        new_chunks: List[Dict[str, Any]],
        current_summary: str = "",
    ) -> str:
        """Generate or refine rolling memory from newly retrieved chunks."""
        context_text = format_chunks_for_llm(new_chunks)
        context_text = truncate_to_tokens(context_text, 4096)
        current_summary = truncate_to_tokens(current_summary, 4096)

        if not current_summary:
            prompt = f"""Please create a concise summary of the following information as it relates to answering this question:

Question: {question}

Information:
{context_text}

Your summary should:
1. Include all relevant facts that might help answer the question
2. Exclude irrelevant information
3. Be clear and concise
4. Preserve specific details, dates, numbers, and names that may be relevant

Summary:"""
        else:
            prompt = f"""Please refine the following information summary using newly retrieved information.

Question: {question}

Current summary:
{current_summary}

New information:
{context_text}

Your refined summary should:
1. Integrate new relevant facts with the existing summary
2. Remove redundancies
3. Remain concise while preserving all important information
4. Prioritize information that helps answer the question
5. Maintain specific details, dates, numbers, and names that may be relevant

Refined summary:"""

        try:
            return llm_chat(prompt, max_tokens=SUMMARY_MAX_TOKENS, expect_json=False)
        except Exception as e:
            print("Error generating/refining summary:", e)
            if current_summary:
                return f"{current_summary}\n\nNew information:\n{context_text}"
            return context_text

    def warm_up_analysis(self, question: str, info_summary: str) -> Dict[str, Any]:
        """Analyze whether warm-up retrieval is enough, and produce dependencies if needed."""
        info_summary = truncate_to_tokens(info_summary, 8192)

        prompt = f"""Question: {question}

Available Information:
{info_summary}

Based on the information provided, please analyze:
1. Can the question be answered completely with this information? (Yes/No)
2. What specific information is missing, if any?
3. What specific question should we ask to find the missing information?
4. Summarize our current understanding based on available information.
5. What are the key dependencies needed to answer this question?
6. Why is information missing? (max 20 words)

Please format your response as a JSON object with these keys:
- "can_answer": boolean
- "missing_info": string
- "subquery": string
- "current_understanding": string
- "dependencies": list of strings (key information dependencies)
- "missing_reason": string (brief explanation why info is missing, max 20 words)"""

        try:
            response = llm_chat(prompt, max_tokens=JSON_MAX_TOKENS, expect_json=True)
            result = parse_json_object(response)

            if result is None:
                return {
                    "can_answer": True,
                    "missing_info": "",
                    "subquery": question,
                    "current_understanding": "Failed to parse warm-up analysis response.",
                    "dependencies": ["Information relevant to the question"],
                    "missing_reason": "Parse error occurred",
                }

            result["can_answer"] = coerce_bool(result.get("can_answer"), default=True)
            result["missing_info"] = str(result.get("missing_info", ""))
            result["subquery"] = str(result.get("subquery") or question)
            result["current_understanding"] = str(result.get("current_understanding", ""))
            result["dependencies"] = normalize_dependencies(result.get("dependencies"), question)
            result["missing_reason"] = str(
                result.get(
                    "missing_reason",
                    "Additional context needed" if not result["can_answer"] else "No missing information",
                )
            )

            return result

        except Exception as e:
            print("Error in warm_up_analysis:", e)
            return {
                "can_answer": True,
                "missing_info": "",
                "subquery": question,
                "current_understanding": f"Error during analysis: {str(e)}",
                "dependencies": ["Information relevant to the question"],
                "missing_reason": "Analysis error occurred",
            }

    def dependency_aware_rag(
        self,
        question: str,
        info_summary: str,
        dependencies: List[str],
        idx: int,
    ) -> Dict[str, Any]:
        """Analyze whether the current dependency is solved by the rolling memory."""
        info_summary = truncate_to_tokens(info_summary, 8192)

        prompt = f"""
            We pre-parsed the question into a list of dependencies, and the dependencies are sorted in a topological order, below is the question, the information summary, and the decomposed dependencies:

            Question: {question}

            Available Information:
            {info_summary}

            Decomposed dependencies:
            {dependencies}

            Current dependency to be answered:
            {dependencies[idx]}

            Please analyze the question and the information summary, and the decomposed dependencies, and answer the following questions:
            Please analyze:
            1. Can the question be answered completely with this information? (Yes/No)
            2. Summarize our current understanding based on available information.

            Please format your response as a JSON object with these keys:
            - "can_answer": boolean
            - "current_understanding": string
            """

        try:
            response = llm_chat(prompt, max_tokens=JSON_MAX_TOKENS, expect_json=True)
            result = parse_json_object(response)

            if result is None:
                return {
                    "can_answer": True,
                    "current_understanding": "Failed to parse dependency-aware response.",
                }

            result["can_answer"] = coerce_bool(result.get("can_answer"), default=True)
            result["current_understanding"] = str(result.get("current_understanding", ""))

            return result

        except Exception as e:
            print("Error in dependency_aware_rag:", e)
            return {
                "can_answer": True,
                "current_understanding": f"Error during analysis: {str(e)}",
            }

    def generate_answer(self, question: str, info_summary: str) -> str:
        """Generate the final answer from the rolling memory only."""
        info_summary = truncate_to_tokens(info_summary, 10000)

        # This prompt matches the repository final-answer prompt structure.
        # The LLM receives question + final summarized evidence memory, not raw evidence_chunk.
        prompt = f"""You must give ONLY the direct answer in the most concise way possible. DO NOT explain or provide any additional context.
If the answer is a simple yes/no, just say "Yes." or "No."
If the answer is a name, just give the name.
If the answer is a date, just give the date.
If the answer is a number, just give the number.
If the answer requires a brief phrase, make it as concise as possible.

Question: {question}

Information Summary:
{info_summary}

Remember: Be concise - give ONLY the essential answer, nothing more.
Ans: """

        try:
            return clean_final_answer(
                llm_chat(prompt, max_tokens=ANSWER_MAX_TOKENS, expect_json=False)
            )
        except Exception as e:
            print("Error generating answer:", e)
            return ""

    def _sort_dependencies(self, dependencies: List[str], query: str) -> List[str]:
        """Ask the LLM for dependency edges and topologically sort dependencies."""
        dependencies = normalize_dependencies(dependencies, query)

        if len(dependencies) <= 1:
            return dependencies

        prompt = f"""
        Given the question:
        Question: {query}

        and its decomposed dependencies:
        Dependencies: {dependencies}

        Please output the dependency pairs that dependency A relies on dependency B, if any. If no dependency pairs are found, output an empty list.

        format your response as a JSON object with these keys:
        - "dependency_pairs": list of tuples of integers
        """

        try:
            response = llm_chat(prompt, max_tokens=JSON_MAX_TOKENS, expect_json=True)
            result = parse_json_object(response)

            if result is None:
                return dependencies

            dependency_pairs = normalize_dependency_pairs(result.get("dependency_pairs", []))
            return self._topological_sort(dependencies, dependency_pairs)

        except Exception as e:
            print("Error sorting dependencies:", e)
            return dependencies

    @staticmethod
    def _topological_sort(
        dependencies: List[str],
        dependency_pairs: List[Tuple[int, int]],
    ) -> List[str]:
        """Use graph-based algorithm to sort dependencies in topological order."""
        graph = {dep: [] for dep in dependencies}

        for dependent_idx, dependency_idx in dependency_pairs:
            if 0 <= dependent_idx < len(dependencies) and 0 <= dependency_idx < len(dependencies):
                dependent = dependencies[dependent_idx]
                dependency = dependencies[dependency_idx]
                graph[dependency].append(dependent)

        visited = set()
        stack = []

        def dfs(node: str) -> None:
            if node in visited:
                return
            visited.add(node)
            for neighbor in graph[node]:
                dfs(neighbor)
            stack.append(node)

        for node in graph:
            if node not in visited:
                dfs(node)

        sorted_dependencies = stack[::-1]
        return sorted_dependencies if sorted_dependencies else dependencies

    @staticmethod
    def _annotate_chunks(
        chunks: List[Dict[str, Any]],
        retrieval_query: str,
        stage: str,
        round_number: int,
    ) -> List[Dict[str, Any]]:
        """Add retrieval metadata to raw chunks for evidence storage."""
        annotated = []

        for chunk in chunks:
            item = dict(chunk)
            item["retrieval_query"] = retrieval_query
            item["stage"] = stage
            item["round"] = int(round_number)
            annotated.append(item)

        return annotated

    @staticmethod
    def _deduplicate_evidence(chunks: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Deduplicate evidence chunks by chunk_id when available."""
        seen = set()
        deduped = []

        for chunk in chunks:
            key = chunk.get("chunk_id") or (chunk.get("title", ""), chunk.get("text", ""))

            if key in seen:
                continue

            seen.add(key)
            deduped.append(chunk)

        return deduped

    def answer_question(self, question: str) -> Dict[str, Any]:
        """Run LogicRAG for one question. Only the question string is accepted here."""
        info_summary = ""
        round_count = 0
        dependency_analysis_history = []
        all_evidence_chunks = []

        # Stage 1: warm-up retrieval.
        warmup_chunks = self.retriever.retrieve(question, top_k=self.top_k)
        warmup_chunks = self._annotate_chunks(
            warmup_chunks,
            retrieval_query=question,
            stage="warmup",
            round_number=0,
        )
        all_evidence_chunks.extend(warmup_chunks)

        info_summary = self.refine_summary_with_context(
            question=question,
            new_chunks=warmup_chunks,
            current_summary=info_summary,
        )

        analysis = self.warm_up_analysis(question, info_summary)

        if analysis["can_answer"]:
            answer = self.generate_answer(question, info_summary)
            self.last_dependency_analysis = []
            return {
                "answer": answer,
                "evidence_chunk": self._deduplicate_evidence(all_evidence_chunks),
                "final_evidence_memory": info_summary,
                "rounds": round_count,
                "dependency_analysis": [],
            }

        sorted_dependencies = self._sort_dependencies(analysis["dependencies"], question)
        dependency_analysis_history.append({"sorted_dependencies": sorted_dependencies})

        # Stage 2: dependency-aware iterative retrieval.
        idx = 0

        while round_count < self.max_rounds and idx < len(sorted_dependencies):
            round_count += 1
            current_query = sorted_dependencies[idx]

            round_chunks = self.retriever.retrieve(current_query, top_k=self.top_k)
            round_chunks = self._annotate_chunks(
                round_chunks,
                retrieval_query=current_query,
                stage="dependency",
                round_number=round_count,
            )
            all_evidence_chunks.extend(round_chunks)

            info_summary = self.refine_summary_with_context(
                question=question,
                new_chunks=round_chunks,
                current_summary=info_summary,
            )

            round_analysis = self.dependency_aware_rag(
                question=question,
                info_summary=info_summary,
                dependencies=sorted_dependencies,
                idx=idx,
            )

            dependency_analysis_history.append({
                "round": round_count,
                "query": current_query,
                "analysis": round_analysis,
            })

            if round_analysis["can_answer"]:
                answer = self.generate_answer(question, info_summary)
                self.last_dependency_analysis = dependency_analysis_history
                return {
                    "answer": answer,
                    "evidence_chunk": self._deduplicate_evidence(all_evidence_chunks),
                    "final_evidence_memory": info_summary,
                    "rounds": round_count,
                    "dependency_analysis": dependency_analysis_history,
                }

            idx += 1

        # If max rounds reached, generate best possible answer.
        answer = self.generate_answer(question, info_summary)
        self.last_dependency_analysis = dependency_analysis_history

        return {
            "answer": answer,
            "evidence_chunk": self._deduplicate_evidence(all_evidence_chunks),
            "final_evidence_memory": info_summary,
            "rounds": round_count,
            "dependency_analysis": dependency_analysis_history,
        }

print("LogicRAG-vLLM class is ready.")

LogicRAG-vLLM class is ready.


In [10]:
#cell 10
# Dataset runner.
# This cell performs NO evaluation.
# It saves:
# 1) evidence/{dataset}_evidence.json
# 2) answer/{dataset}_qwen3.5_answers.json
#
# Only the question field is passed to LogicRAGVLLM.
# type/answer/supports are copied only after generation for output formatting.
#
# IMPORTANT:
# If an existing evidence file does not contain final_evidence_memory,
# this cell backs up the old files and restarts that dataset from zero,
# because the memory cannot be reconstructed without rerunning LogicRAG.

def load_shuffled_records_for_run(dataset_name: str) -> List[Dict[str, Any]]:
    """Load the shuffled local question records and apply optional start/end slicing."""
    cfg = DATASETS[dataset_name]

    records = load_json(cfg["question_local_shuffled_path"])
    if not isinstance(records, list):
        raise ValueError(f"{dataset_name}: shuffled question file root must be a list.")

    records = records[QUESTION_START_INDEX:QUESTION_END_INDEX]

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Records selected after shuffle:", len(records))
    print("Question start index:", QUESTION_START_INDEX)
    print("Question end index:", QUESTION_END_INDEX)

    return records

def backup_file_if_exists(path: Path) -> Optional[Path]:
    """Create a timestamped backup of an existing file."""
    if not path.exists():
        return None

    timestamp = time.strftime("%Y%m%d_%H%M%S")
    backup_path = path.with_name(path.stem + f".backup_{timestamp}" + path.suffix)
    shutil.copy2(path, backup_path)
    return backup_path

def existing_evidence_has_memory(evidence: List[Dict[str, Any]]) -> bool:
    """Check whether all existing evidence records already contain final_evidence_memory."""
    if not evidence:
        return True

    return all(
        isinstance(row, dict) and "final_evidence_memory" in row
        for row in evidence
    )

def load_or_initialize_outputs(
    dataset_name: str,
    answer_path: Path,
    evidence_path: Path,
) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]], int]:
    """Load existing outputs for resume, and align answer/evidence lengths."""
    if not RESUME_IF_EXISTS:
        return [], [], 0

    answers = load_json_if_exists(answer_path, [])
    evidence = load_json_if_exists(evidence_path, [])

    if not isinstance(answers, list):
        print(f"Warning: existing answer file is not a list. Starting fresh: {answer_path}")
        answers = []

    if not isinstance(evidence, list):
        print(f"Warning: existing evidence file is not a list. Starting fresh: {evidence_path}")
        evidence = []

    # If old evidence was created before final_evidence_memory was added,
    # restart the dataset because the missing memory cannot be recovered exactly.
    if evidence and not existing_evidence_has_memory(evidence):
        answer_backup = backup_file_if_exists(answer_path)
        evidence_backup = backup_file_if_exists(evidence_path)

        print("=" * 100)
        print(f"{dataset_name}: existing evidence file does not contain final_evidence_memory.")
        print("Restarting this dataset from zero to create consistent evidence outputs.")

        if answer_backup is not None:
            print("Answer backup:", answer_backup)
        if evidence_backup is not None:
            print("Evidence backup:", evidence_backup)

        return [], [], 0

    resume_n = min(len(answers), len(evidence))

    if len(answers) != len(evidence):
        print(
            f"Warning: answer/evidence length mismatch for {dataset_name}. "
            f"Trimming to {resume_n}."
        )
        answers = answers[:resume_n]
        evidence = evidence[:resume_n]
        atomic_save_json(answers, answer_path)
        atomic_save_json(evidence, evidence_path)

    if resume_n > 0:
        print(f"Resuming {dataset_name} from record index {resume_n}.")
        print("Existing records already contain final_evidence_memory.")

    return answers, evidence, resume_n

def run_dataset(dataset_name: str) -> Dict[str, Any]:
    """Run LogicRAG on one dataset completely separately."""
    cfg = DATASETS[dataset_name]

    records = load_shuffled_records_for_run(dataset_name)

    answer_path = cfg["answer_output_path"]
    evidence_path = cfg["evidence_output_path"]

    answer_records, evidence_records, start_index = load_or_initialize_outputs(
        dataset_name=dataset_name,
        answer_path=answer_path,
        evidence_path=evidence_path,
    )

    retriever = Stage1EmbeddingRetriever(
        corpus_path=cfg["corpus_local_path"],
        embedding_path=cfg["embedding_local_path"],
        query_encoder=query_encoder,
        top_k=TOP_K,
    )

    rag = LogicRAGVLLM(
        retriever=retriever,
        top_k=TOP_K,
        max_rounds=MAX_ROUNDS,
    )

    pbar = tqdm(
        range(start_index, len(records)),
        desc=f"{dataset_name} LogicRAG",
        dynamic_ncols=True,
    )

    for i in pbar:
        item = records[i]

        # CRITICAL: only this string is passed to the LLM pipeline.
        question = item["question"]

        try:
            result = rag.answer_question(question)
            response = clean_final_answer(result.get("answer", ""))
            evidence_chunk = result.get("evidence_chunk", [])
            final_evidence_memory = result.get("final_evidence_memory", "")

        except Exception as e:
            print("=" * 100)
            print(f"Error on dataset={dataset_name}, shuffled_index={i}")
            print("Question:", question)
            print("Exception:", e)
            print(traceback.format_exc())

            response = ""
            evidence_chunk = []
            final_evidence_memory = ""

        answer_records.append({
            "type": item.get("type", ""),
            "question": question,
            "gt": item.get("answer", ""),
            "response": response,
        })

        evidence_records.append({
            "type": item.get("type", ""),
            "question": question,
            "answer": item.get("answer", ""),
            "supports": item.get("supports", []),
            "evidence_chunk": evidence_chunk,
            "final_evidence_memory": final_evidence_memory,
        })

        processed = i + 1
        pbar.set_postfix({
            "saved": len(answer_records),
            "chunks": len(evidence_chunk),
            "memory_tokens": count_llm_tokens(final_evidence_memory) if final_evidence_memory else 0,
        })

        if processed % SAVE_EVERY_N == 0:
            atomic_save_json(answer_records, answer_path)
            atomic_save_json(evidence_records, evidence_path)
            print(f"Saved checkpoint at {processed} records for {dataset_name}.")
            print("Answer file:", answer_path)
            print("Evidence file:", evidence_path)

        if processed % CLEAR_CACHE_EVERY_N == 0:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    atomic_save_json(answer_records, answer_path)
    atomic_save_json(evidence_records, evidence_path)

    print("=" * 100)
    print("Finished dataset:", dataset_name)
    print("Total answer records:", len(answer_records))
    print("Total evidence records:", len(evidence_records))
    print("Answer file:", answer_path)
    print("Evidence file:", evidence_path)

    # Free dataset-specific retrieval memory before the next dataset.
    del rag
    del retriever
    gc.collect()

    return {
        "dataset": dataset_name,
        "num_answers": len(answer_records),
        "num_evidence": len(evidence_records),
        "answer_path": str(answer_path),
        "evidence_path": str(evidence_path),
    }

print("Dataset runner is ready.")
print("Evidence records will now include final_evidence_memory.")

Dataset runner is ready.
Evidence records will now include final_evidence_memory.


In [ ]:
#cell 11
# Run HotpotQA separately.

hotpotqa_run_summary = run_dataset("hotpotqa")
hotpotqa_run_summary

Dataset: hotpotqa
Records selected after shuffle: 1000
Question start index: 0
Question end index: None
Retriever initialized.
Corpus: /content/final_project_logicrag_stage2/corpus/hotpotqa_logicrag_corpus.json
Embedding: /content/final_project_logicrag_stage2/embeddings/hotpotqa_embeddings.pt
Number of chunks: 35029
Embedding shape: (35029, 384)
Top-k: 5


hotpotqa LogicRAG:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved checkpoint at 10 records for hotpotqa.
Answer file: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_qwen3.5_answers.json
Evidence file: /content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json
Saved checkpoint at 20 records for hotpotqa.
Answer file: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_qwen3.5_answers.json
Evidence file: /content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json
Saved checkpoint at 30 records for hotpotqa.
Answer file: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_qwen3.5_answers.json
Evidence file: /content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json
Saved checkpoint at 40 records for hotpotqa.
Answer file: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_qwen3.5_answers.json
Evidence file: /content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json
Saved checkpoint at 50 records for hotpotqa.
Answer file: /conte

{'dataset': 'hotpotqa',
 'num_answers': 1000,
 'num_evidence': 1000,
 'answer_path': '/content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_qwen3.5_answers.json',
 'evidence_path': '/content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json'}

In [ ]:
#cell 12
# Run 2WikiMultiHopQA separately.

wikimultihopqa_run_summary = run_dataset("2wikimultihopqa")
wikimultihopqa_run_summary

Dataset: 2wikimultihopqa
Records selected after shuffle: 1000
Question start index: 0
Question end index: None
Retriever initialized.
Corpus: /content/final_project_logicrag_stage2/corpus/2wikimultihopqa_logicrag_corpus.json
Embedding: /content/final_project_logicrag_stage2/embeddings/2wikimultihopqa_embeddings.pt
Number of chunks: 12685
Embedding shape: (12685, 384)
Top-k: 5


2wikimultihopqa LogicRAG:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved checkpoint at 10 records for 2wikimultihopqa.
Answer file: /content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_qwen3.5_answers.json
Evidence file: /content/drive/MyDrive/final_project/logicRAG/evidence/2wikimultihopqa_evidence.json
Saved checkpoint at 20 records for 2wikimultihopqa.
Answer file: /content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_qwen3.5_answers.json
Evidence file: /content/drive/MyDrive/final_project/logicRAG/evidence/2wikimultihopqa_evidence.json
Saved checkpoint at 30 records for 2wikimultihopqa.
Answer file: /content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_qwen3.5_answers.json
Evidence file: /content/drive/MyDrive/final_project/logicRAG/evidence/2wikimultihopqa_evidence.json
Saved checkpoint at 40 records for 2wikimultihopqa.
Answer file: /content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_qwen3.5_answers.json
Evidence file: /content/drive/MyDrive/final_project/logicRAG/evidence/2wikimulti

{'dataset': '2wikimultihopqa',
 'num_answers': 1000,
 'num_evidence': 1000,
 'answer_path': '/content/drive/MyDrive/final_project/logicRAG/answer/2wikimultihopqa_qwen3.5_answers.json',
 'evidence_path': '/content/drive/MyDrive/final_project/logicRAG/evidence/2wikimultihopqa_evidence.json'}

In [11]:
#cell 13
# Validate final output files and show a small preview.
# This does not evaluate correctness.

def validate_final_outputs(dataset_name: str) -> None:
    """Validate saved answer/evidence file shapes and keys."""
    cfg = DATASETS[dataset_name]

    answers = load_json(cfg["answer_output_path"])
    evidence = load_json(cfg["evidence_output_path"])

    if not isinstance(answers, list):
        raise ValueError(f"{dataset_name}: answer output is not a list.")

    if not isinstance(evidence, list):
        raise ValueError(f"{dataset_name}: evidence output is not a list.")

    if len(answers) != len(evidence):
        raise ValueError(
            f"{dataset_name}: answer/evidence length mismatch: "
            f"{len(answers)} vs {len(evidence)}"
        )

    expected_answer_keys = {"type", "question", "gt", "response"}
    expected_evidence_keys = {
        "type",
        "question",
        "answer",
        "supports",
        "evidence_chunk",
        "final_evidence_memory",
    }

    if answers:
        missing = expected_answer_keys - set(answers[0].keys())
        if missing:
            raise ValueError(f"{dataset_name}: first answer record missing keys: {missing}")

    if evidence:
        missing = expected_evidence_keys - set(evidence[0].keys())
        if missing:
            raise ValueError(f"{dataset_name}: first evidence record missing keys: {missing}")

    print("=" * 100)
    print("Dataset:", dataset_name)
    print("Answer records:", len(answers))
    print("Evidence records:", len(evidence))
    print("Answer path:", cfg["answer_output_path"])
    print("Evidence path:", cfg["evidence_output_path"])

    if answers:
        print("\nFirst answer record:")
        print(json.dumps(answers[0], ensure_ascii=False, indent=2)[:2000])

    if evidence:
        print("\nFirst evidence record preview:")
        preview = dict(evidence[0])

        # Keep preview small.
        preview["evidence_chunk"] = preview["evidence_chunk"][:2]

        print(json.dumps(preview, ensure_ascii=False, indent=2)[:5000])

        print("\nFirst final_evidence_memory:")
        print(preview.get("final_evidence_memory", "")[:3000])

for dataset_name in DATASET_RUN_ORDER:
    validate_final_outputs(dataset_name)

print("\nFinal files:")
for dataset_name, cfg in DATASETS.items():
    print("=" * 100)
    print(dataset_name)
    print("Evidence:", cfg["evidence_output_path"])
    print("Answer:", cfg["answer_output_path"])

Dataset: hotpotqa
Answer records: 1000
Evidence records: 1000
Answer path: /content/drive/MyDrive/final_project/logicRAG/answer/hotpotqa_qwen3.5_answers.json
Evidence path: /content/drive/MyDrive/final_project/logicRAG/evidence/hotpotqa_evidence.json

First answer record:
{
  "type": "comparison",
  "question": "George Gershwin is an American Composer and Judith Weir is a composer from which country?",
  "gt": "a British composer",
  "response": "Britain"
}

First evidence record preview:
{
  "type": "comparison",
  "question": "George Gershwin is an American Composer and Judith Weir is a composer from which country?",
  "answer": "a British composer",
  "supports": [
    [
      "Judith Weir",
      "Judith Weir {'1': \", '2': \", '3': \", '4': \"} (born 11 May 1954) is a British composer and Master of the Queen's Music."
    ],
    [
      "George Gershwin",
      "George Jacob Gershwin ( ; September 26, 1898 July 11, 1937) was an American composer and pianist."
    ]
  ],
  "evidenc